In [ ]:
# ── Cell 1: Set Random Seeds for Reproducibility ─────────────────────

import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import joblib
import os

# Fix all random seeds to ensure reproducible results
# Using the same seed as LSTM for fair comparison
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Random seed : {SEED}")
print(f"Device      : {device}")

In [ ]:
# ── Cell 2: Load Preprocessed Data ───────────────────────────────────

# Auto-detect environment (Colab or Local)
# Same data as LSTM notebook - ensures fair comparison between models
try:
    from google.colab import drive
    drive.mount('/content/drive')
    COLAB = True
except ImportError:
    COLAB = False

if COLAB and os.path.exists('/content/drive/MyDrive/Machine_Learning'):
    DATA_PATH    = '/content/drive/MyDrive/Machine_Learning/data/'
    MODELS_PATH  = '/content/drive/MyDrive/Machine_Learning/models/'
    RESULTS_PATH = '/content/drive/MyDrive/Machine_Learning/results/'
    print("Environment: Google Colab + Google Drive")
else:
    DATA_PATH    = '../data/'
    MODELS_PATH  = '../models/'
    RESULTS_PATH = '../results/'
    print("Environment: Local")

print(f"Data path    : {DATA_PATH}")
print(f"Models path  : {MODELS_PATH}")
print(f"Results path : {RESULTS_PATH}")

# Load Site 2078 (Southbound)
X_train_2078 = np.load(DATA_PATH + 'X_train_2078.npy')
y_train_2078 = np.load(DATA_PATH + 'y_train_2078.npy')
X_val_2078   = np.load(DATA_PATH + 'X_val_2078.npy')
y_val_2078   = np.load(DATA_PATH + 'y_val_2078.npy')
X_test_2078  = np.load(DATA_PATH + 'X_test_2078.npy')
y_test_2078  = np.load(DATA_PATH + 'y_test_2078.npy')

# Load Site 2079 (Northbound)
X_train_2079 = np.load(DATA_PATH + 'X_train_2079.npy')
y_train_2079 = np.load(DATA_PATH + 'y_train_2079.npy')
X_val_2079   = np.load(DATA_PATH + 'X_val_2079.npy')
y_val_2079   = np.load(DATA_PATH + 'y_val_2079.npy')
X_test_2079  = np.load(DATA_PATH + 'X_test_2079.npy')
y_test_2079  = np.load(DATA_PATH + 'y_test_2079.npy')

# Load scalers for inverse transformation
scaler_2078 = joblib.load(MODELS_PATH + 'scaler_2078.pkl')
scaler_2079 = joblib.load(MODELS_PATH + 'scaler_2079.pkl')

print("\nData loaded successfully!")
print(f"\nSite 2078:")
print(f"  X_train: {X_train_2078.shape}, y_train: {y_train_2078.shape}")
print(f"  X_val  : {X_val_2078.shape},   y_val  : {y_val_2078.shape}")
print(f"  X_test : {X_test_2078.shape},  y_test : {y_test_2078.shape}")
print(f"\nSite 2079:")
print(f"  X_train: {X_train_2079.shape}, y_train: {y_train_2079.shape}")
print(f"  X_val  : {X_val_2079.shape},   y_val  : {y_val_2079.shape}")
print(f"  X_test : {X_test_2079.shape},  y_test : {y_test_2079.shape}")

In [ ]:
# ── Cell 3: Convert to PyTorch Tensors and Create DataLoaders ─────────

BATCH_SIZE = 64

def make_loaders(X_train, y_train, X_val, y_val, X_test, y_test, batch_size):
    # Convert numpy arrays to PyTorch tensors
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32)
    X_val_t   = torch.tensor(X_val,   dtype=torch.float32)
    y_val_t   = torch.tensor(y_val,   dtype=torch.float32)
    X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
    y_test_t  = torch.tensor(y_test,  dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(X_train_t, y_train_t),
                              batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(TensorDataset(X_val_t, y_val_t),
                              batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(TensorDataset(X_test_t, y_test_t),
                              batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

train_loader_2078, val_loader_2078, test_loader_2078 = make_loaders(
    X_train_2078, y_train_2078, X_val_2078, y_val_2078,
    X_test_2078, y_test_2078, BATCH_SIZE)

train_loader_2079, val_loader_2079, test_loader_2079 = make_loaders(
    X_train_2079, y_train_2079, X_val_2079, y_val_2079,
    X_test_2079, y_test_2079, BATCH_SIZE)

print(f"Batch size: {BATCH_SIZE}")
print(f"\nSite 2078:")
print(f"  Train batches : {len(train_loader_2078)}")
print(f"  Val batches   : {len(val_loader_2078)}")
print(f"  Test batches  : {len(test_loader_2078)}")

In [ ]:
# ── Cell 4: Define GRU Model ──────────────────────────────────────────

# GRU (Gated Recurrent Unit) is a simplified version of LSTM.
# Key differences from LSTM:
#   - LSTM has 3 gates (input, forget, output) + cell state
#   - GRU has 2 gates (update, reset) only, no separate cell state
#   - GRU has fewer parameters → faster training, less memory usage
#   - Both are designed to capture long-term temporal dependencies

class GRUModel(nn.Module):
    """
    GRU model for time series forecasting.

    Architecture:
        Input → GRU layers → Dropout → Fully Connected → Output

    Args:
        input_size  : number of input features (1 for univariate time series)
        hidden_size : number of hidden units in each GRU layer
        num_layers  : number of stacked GRU layers
        dropout     : dropout rate between GRU layers (regularisation)
        output_size : number of output values (1 for single-step prediction)
    """
    def __init__(self, input_size=1, hidden_size=64,
                 num_layers=2, dropout=0.2, output_size=1):
        super(GRUModel, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        # GRU layer: unlike LSTM, GRU only needs hidden state (no cell state)
        # batch_first=True means input shape is (batch, seq_len, features)
        self.gru = nn.GRU(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout
        )

        # Dropout layer after GRU to prevent overfitting
        self.dropout = nn.Dropout(dropout)

        # Fully connected output layer
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # GRU only needs hidden state h0 (no cell state c0 like LSTM)
        h0 = torch.zeros(self.num_layers, x.size(0),
                         self.hidden_size).to(x.device)

        # Forward pass through GRU
        out, _ = self.gru(x, h0)  # out shape: (batch, seq_len, hidden_size)

        # Take only the last time step output
        out = self.dropout(out[:, -1, :])  # Shape: (batch, hidden_size)

        # Map to output
        out = self.fc(out)                 # Shape: (batch, 1)
        return out

# Initialise GRU model with same hyperparameters as LSTM for fair comparison
model_gru = GRUModel(
    input_size  = 1,
    hidden_size = 64,   # Same as LSTM
    num_layers  = 2,    # Same as LSTM
    dropout     = 0.2,  # Same as LSTM
    output_size = 1
).to(device)

print(model_gru)
print(f"\nTotal trainable parameters: "
      f"{sum(p.numel() for p in model_gru.parameters() if p.requires_grad):,}")

In [ ]:
# ── Cell 5: Define Training Functions ────────────────────────────────

# Same loss function and optimiser as LSTM for fair comparison
criterion = nn.MSELoss()
optimiser_gru = torch.optim.Adam(model_gru.parameters(), lr=0.001)

def train_one_epoch(model, loader, criterion, optimiser, device):
    """Run one full pass through training data and update model weights."""
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        optimiser.zero_grad()
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimiser.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion, device):
    """Evaluate model without updating weights."""
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            total_loss += loss.item()
    return total_loss / len(loader)

print("Training functions defined!")
print(f"  Loss function : MSELoss")
print(f"  Optimiser     : Adam (lr=0.001)")

In [ ]:
# ── Cell 6: Train GRU Model on Site 2078 ─────────────────────────────

N_EPOCHS = 20
PATIENCE = 5

train_losses_2078 = []
val_losses_2078   = []

best_val_loss      = float('inf')
patience_count     = 0
best_model_state   = None

print("Training GRU on Site 2078 (Southbound)...")
print(f"{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>12} {'Status':>10}")
print("-" * 45)

for epoch in range(1, N_EPOCHS + 1):
    train_loss = train_one_epoch(model_gru, train_loader_2078,
                                 criterion, optimiser_gru, device)
    val_loss   = evaluate(model_gru, val_loader_2078, criterion, device)

    train_losses_2078.append(train_loss)
    val_losses_2078.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_count   = 0
        best_model_state = model_gru.state_dict().copy()
        status = "✓ improved"
    else:
        patience_count += 1
        status = f"no improve ({patience_count}/{PATIENCE})"

    print(f"{epoch:>6} {train_loss:>12.6f} {val_loss:>12.6f} {status:>10}")

    if patience_count >= PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch}")
        break

model_gru.load_state_dict(best_model_state)
print(f"\nTraining complete! Best val loss: {best_val_loss:.6f}")
torch.save(model_gru.state_dict(), MODELS_PATH + 'gru_2078.pt')
print(f"Model saved to {MODELS_PATH}gru_2078.pt")

In [ ]:
# ── Cell 7: Train GRU Model on Site 2079 ─────────────────────────────

# Re-initialise model and optimiser for Site 2079
torch.manual_seed(SEED)
model_gru_2079 = GRUModel(
    input_size  = 1,
    hidden_size = 64,
    num_layers  = 2,
    dropout     = 0.2,
    output_size = 1
).to(device)

optimiser_gru_2079 = torch.optim.Adam(model_gru_2079.parameters(), lr=0.001)

train_losses_2079 = []
val_losses_2079   = []

best_val_loss_2079    = float('inf')
patience_count_2079   = 0
best_model_state_2079 = None

print("Training GRU on Site 2079 (Northbound)...")
print(f"{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>12} {'Status':>10}")
print("-" * 45)

for epoch in range(1, N_EPOCHS + 1):
    train_loss = train_one_epoch(model_gru_2079, train_loader_2079,
                                 criterion, optimiser_gru_2079, device)
    val_loss   = evaluate(model_gru_2079, val_loader_2079, criterion, device)

    train_losses_2079.append(train_loss)
    val_losses_2079.append(val_loss)

    if val_loss < best_val_loss_2079:
        best_val_loss_2079    = val_loss
        patience_count_2079   = 0
        best_model_state_2079 = model_gru_2079.state_dict().copy()
        status = "✓ improved"
    else:
        patience_count_2079 += 1
        status = f"no improve ({patience_count_2079}/{PATIENCE})"

    print(f"{epoch:>6} {train_loss:>12.6f} {val_loss:>12.6f} {status:>10}")

    if patience_count_2079 >= PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch}")
        break

model_gru_2079.load_state_dict(best_model_state_2079)
print(f"\nTraining complete! Best val loss: {best_val_loss_2079:.6f}")
torch.save(model_gru_2079.state_dict(), MODELS_PATH + 'gru_2079.pt')
print(f"Model saved to {MODELS_PATH}gru_2079.pt")

In [ ]:
# ── Cell 8: Evaluate GRU Model on Test Set ───────────────────────────

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_model(model, X_test, y_test, scaler, site_name, device, batch_size=256):
    """Evaluate trained model on test set using batches to avoid GPU OOM."""
    model.eval()
    all_preds = []
    for i in range(0, len(X_test), batch_size):
        X_batch = torch.tensor(
            X_test[i:i+batch_size], dtype=torch.float32).to(device)
        with torch.no_grad():
            preds = model(X_batch).cpu().numpy()
        all_preds.append(preds)

    y_pred_scaled = np.concatenate(all_preds, axis=0)
    y_pred = scaler.inverse_transform(y_pred_scaled)
    y_true = scaler.inverse_transform(y_test)

    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true, y_pred)

    print(f"\n=== {site_name} - Test Set Metrics ===")
    print(f"  MAE  : {mae:.4f}")
    print(f"  MSE  : {mse:.4f}")
    print(f"  RMSE : {rmse:.4f}")
    print(f"  R²   : {r2:.4f}")

    return y_pred, y_true, {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}

# Evaluate Site 2078
model_gru.load_state_dict(torch.load(MODELS_PATH + 'gru_2078.pt',
                                      map_location=device))
y_pred_2078, y_true_2078, metrics_2078 = evaluate_model(
    model_gru, X_test_2078, y_test_2078, scaler_2078,
    'Site 2078 (Southbound)', device)

# Evaluate Site 2079
model_gru_2079.load_state_dict(torch.load(MODELS_PATH + 'gru_2079.pt',
                                           map_location=device))
y_pred_2079, y_true_2079, metrics_2079 = evaluate_model(
    model_gru_2079, X_test_2079, y_test_2079, scaler_2079,
    'Site 2079 (Northbound)', device)

In [ ]:
# ── Cell 9: Plot Learning Curves ──────────────────────────────────────

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, len(train_losses_2078)+1), train_losses_2078,
             label='Train Loss', color='steelblue', linewidth=2)
axes[0].plot(range(1, len(val_losses_2078)+1), val_losses_2078,
             label='Val Loss', color='coral', linewidth=2)
axes[0].set_title('Learning Curve - Site 2078 (Southbound)',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].legend()

axes[1].plot(range(1, len(train_losses_2079)+1), train_losses_2079,
             label='Train Loss', color='steelblue', linewidth=2)
axes[1].plot(range(1, len(val_losses_2079)+1), val_losses_2079,
             label='Val Loss', color='coral', linewidth=2)
axes[1].set_title('Learning Curve - Site 2079 (Northbound)',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss (MSE)')
axes[1].legend()

plt.suptitle('GRU Learning Curves - SH1 Green Lane East',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_PATH + 'gru_learning_curves.png', dpi=150)
plt.show()
print("Figure saved!")

In [ ]:
# ── Cell 10: Plot Predictions vs Actual Values ────────────────────────

PLOT_STEPS = 672  # 7 days × 96 steps/day

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

axes[0].plot(y_true_2078[:PLOT_STEPS],
             label='Actual', color='steelblue', linewidth=1)
axes[0].plot(y_pred_2078[:PLOT_STEPS],
             label='Predicted', color='coral', linewidth=1, alpha=0.8)
axes[0].set_title(f'Site 2078 (Southbound) - Predicted vs Actual '
                  f'(First 7 Days of Test Set)\n'
                  f'MAE={metrics_2078["MAE"]:.2f}, '
                  f'RMSE={metrics_2078["RMSE"]:.2f}, '
                  f'R²={metrics_2078["R2"]:.4f}',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Time Steps (15-min intervals)')
axes[0].set_ylabel('Vehicle Count')
axes[0].legend()

axes[1].plot(y_true_2079[:PLOT_STEPS],
             label='Actual', color='steelblue', linewidth=1)
axes[1].plot(y_pred_2079[:PLOT_STEPS],
             label='Predicted', color='coral', linewidth=1, alpha=0.8)
axes[1].set_title(f'Site 2079 (Northbound) - Predicted vs Actual '
                  f'(First 7 Days of Test Set)\n'
                  f'MAE={metrics_2079["MAE"]:.2f}, '
                  f'RMSE={metrics_2079["RMSE"]:.2f}, '
                  f'R²={metrics_2079["R2"]:.4f}',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Time Steps (15-min intervals)')
axes[1].set_ylabel('Vehicle Count')
axes[1].legend()

plt.suptitle('GRU - Predicted vs Actual Traffic Flow - SH1 Green Lane East',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_PATH + 'gru_predictions.png', dpi=150)
plt.show()
print("Figure saved!")

In [ ]:
# ── Cell 11: Scatter Plot - Predicted vs Actual ───────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, y_true, y_pred, metrics, site_name, color in zip(
    axes,
    [y_true_2078, y_true_2079],
    [y_pred_2078, y_pred_2079],
    [metrics_2078, metrics_2079],
    ['Site 2078 (Southbound)', 'Site 2079 (Northbound)'],
    ['steelblue', 'coral']
):
    ax.scatter(y_true[:5000], y_pred[:5000],
               alpha=0.1, s=1, color=color)
    max_val = max(y_true.max(), y_pred.max())
    ax.plot([0, max_val], [0, max_val],
            'r--', linewidth=1.5, label='Perfect Prediction')
    ax.set_title(f'{site_name}\nR²={metrics["R2"]:.4f}',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Actual Vehicle Count')
    ax.set_ylabel('Predicted Vehicle Count')
    ax.legend()

plt.suptitle('GRU - Predicted vs Actual Scatter Plot - SH1 Green Lane East',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_PATH + 'gru_scatter.png', dpi=150)
plt.show()
print("Figure saved!")